# Build `law_2026` — Chunking → Embedding → Qdrant (Kaggle GPU)

Notebook này nhận **output_json/** (các văn bản đã parse sạch bằng `mass_crawler.py` ở máy local)
và thực hiện 3 việc trên Kaggle:

1. **Chunk** theo **Điều** → đúng format `corpus_clean.json`
   (`id`, `text`, `metadata.{unique_article_id, article_id, doc_number, title, context, clause_num}`).
2. **Embed** bằng `AITeamVN/Vietnamese_Embedding` (1024-d, `normalize_embeddings=True`).
3. **Upsert** sang collection **`law_2026`** (distance = **DOT**, GIỐNG HỆT `vietnamese_laws`).

### Chuẩn bị trước khi chạy
- Nén thư mục `output_json/` ở local rồi **upload làm Kaggle Dataset** (vd dataset `law-output-json`),
  sửa `INPUT_DIR` ở Cell *Config* cho khớp đường dẫn mount `/kaggle/input/...`.
- Vào **Settings → Add-ons → Secrets**, thêm `QDRANT_URL` và `QDRANT_API_KEY`.
- Bật **GPU** (T4) trong Settings.

### An toàn / chống ghi đè
- Collection đích là **`law_2026`** — KHÔNG đụng `vietnamese_laws`.
- id điểm = `uuid5(unique_article_id)` ⇒ chạy lại notebook **không nhân đôi** dữ liệu (idempotent upsert).
- Để xoá-tạo lại từ đầu: đặt `RECREATE = True` ở Cell *Config* (mặc định `False`).


## Cell 1 — Cài thư viện

In [ ]:
!pip install -q qdrant-client sentence-transformers python-dotenv tqdm

## Cell 2 — Config (sửa đường dẫn dataset ở đây)

In [ ]:
import os
# PHẢI đặt TRƯỚC khi import torch/CUDA -> chống phân mảnh VRAM (giảm OOM).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re, json, uuid, glob

# >>> SỬA cho khớp dataset bạn upload (chứa thư mục output_json) <<<
INPUT_DIR     = "/kaggle/input/law-output-json/output_json"   # nơi chứa *.json đã parse
WORK_DIR      = "/kaggle/working"

COLLECTION_NAME = "law_2026"       # collection MỚI, tách biệt vietnamese_laws
RECREATE        = False            # True = xoá rồi tạo lại collection từ đầu

EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"
EMBEDDING_DIM   = 1024
QDRANT_DISTANCE = "Dot"            # khớp collection cũ (vector đã normalize)

# --- Chống CUDA OOM trên T4 (14.5GB) ---
# bge-m3 mặc định max_seq_length=8192 -> attention ∝ batch * seq^2 -> nổ VRAM.
# Chốt 2048 (chunk ≤3500 ký tự < 2048 token nên KHÔNG mất chữ) + batch nhỏ.
MAX_SEQ_LEN     = 2048
BATCH_SIZE      = 8                # T4: 8 an toàn. GPU khỏe có thể nâng 16-32.
UUID_NAMESPACE  = uuid.UUID("12345678-1234-5678-1234-567812345678")  # cố định -> idempotent

# Đầu ra ghi ra /kaggle/working để tải về
CORPUS_OUT   = os.path.join(WORK_DIR, "corpus_clean.json")
MANIFEST_OUT = os.path.join(WORK_DIR, "law_manifest.json")
REVIEW_OUT   = os.path.join(WORK_DIR, "doc_number_review.csv")

# Điều dài hơn ngưỡng này -> TÁCH theo Khoản (tránh embedding bị cắt cụt ở 2048 token).
# AITeamVN/Vietnamese_Embedding (bge-m3) max ~2048 token ≈ 5-7k ký tự tiếng Việt.
# 3500 ký tự là an toàn dưới ngưỡng đó mà vẫn giữ đa số Điều nguyên vẹn.
MAX_ARTICLE_CHARS = 3500

assert os.path.isdir(INPUT_DIR), f"Không thấy {INPUT_DIR} — sửa INPUT_DIR cho khớp dataset."
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))
print(f"Tìm thấy {len(files)} file JSON trong {INPUT_DIR}")

## Cell 3 — Nạp Qdrant secrets

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    os.environ["QDRANT_URL"]     = sec.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = sec.get_secret("QDRANT_API_KEY")
    print("Secrets loaded từ Kaggle.")
except Exception as e:
    print("Không load được Kaggle secrets:", e)
    # Fallback (KHÔNG commit notebook public sau khi điền):
    # os.environ["QDRANT_URL"] = "https://..."
    # os.environ["QDRANT_API_KEY"] = "..."

## Cell 4 — Hàm chunking (theo Điều, đúng format corpus_clean)

- `text` = `"<tiêu đề Điều>\n<nội dung Điều (gộp Khoản/Điểm)>"` — giống corpus gốc.
- `metadata.context.chapter` = `"Chương X <tiêu đề>"` (ngữ cảnh để rerank/đọc), `part`/`section` = `None`.
- `clause_num` = `None` cho Điều ngắn; Điều **dài > MAX_ARTICLE_CHARS** được tách theo **Khoản**
  (id có hậu tố `_K<số khoản>`, `clause_num` = số khoản đầu mảnh) để tránh embedding bị cắt cụt.
  `article_id` vẫn là `"Điều N"` nên việc chấm recall theo (văn bản, Điều) không đổi.
- Số hiệu (`doc_number`) suy từ `so_hieu` → `ten_luat` → tên file, kèm cờ *confidence* để rà.


In [ ]:
DOC_NUMBER_RE = re.compile(
    r"(\d+[a-zA-Z]?\s*/\s*\d{2,4}\s*/\s*[A-Za-z0-9Đđ][A-Za-z0-9Đđ\-]*"  # x/yyyy/CODE (NĐ-CP, QH15, TT-BTC)
    r"|\d+\s*/\s*[A-Za-zĐđ]+\-[A-Za-zĐđ]+"       # x/QĐ-TTg
    r"|\d+\s*\-\s*CP)"                                             # 74-CP
)

def normalize_doc_number(raw: str) -> str:
    if not raw:
        return ""
    s = re.sub(r"\s+", "", raw.strip())
    s = re.sub(r"[A-Za-zĐđ]+", lambda m: m.group(0).upper(), s)
    return s

def extract_doc_number(law_json: dict, filename: str):
    for src, conf in [(law_json.get("so_hieu", ""), "high"),
                      (law_json.get("ten_luat", ""), "medium"),
                      (filename.replace("-", "/").replace(".json", ""), "low")]:
        m = DOC_NUMBER_RE.search(str(src))
        if m:
            return normalize_doc_number(m.group(1)), conf
    return normalize_doc_number(law_json.get("so_hieu", "")) or filename.replace(".json", ""), "low"

def infer_document_type(dn: str, name: str = "") -> str:
    dn = dn.upper(); name = (name or "").lower()
    if "/QH" in dn:
        return "Nghị quyết" if name[:15].startswith("nghị quyết") or "nghị quyết" in name[:15] else "Luật"
    if "NĐ-CP" in dn or "ND-CP" in dn or dn.endswith("-CP"): return "Nghị định"
    if "QĐ-TTG" in dn or "QĐ-TTG" in dn.upper() or "/QĐ" in dn or "/QD" in dn: return "Quyết định"
    if dn.startswith("TT") or "/TT-" in dn or re.search(r"/TT[A-ZĐ]", dn): return "Thông tư"
    if "/NQ" in dn: return "Nghị quyết"
    return "Văn bản"

def clean_law_name(ten_luat: str) -> str:
    s = ten_luat or ""
    s = re.sub(r"\s*[-–]\s*Toàn văn.*$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*\(\s*số[^)]*\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*mới nhất.*$", "", s, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", s).strip()

def build_btc_string(dn, dtype, name): return f"{dn}|{dtype} {dn} {name}"

def article_blocks(dieu: dict):
    """Trả (intro, blocks) — intro = noi_dung_chung; blocks = [(khoan_num|None, text)]."""
    intro = (dieu.get("noi_dung_chung") or "").strip()
    blocks = []
    for k in dieu.get("khoan", []):
        kid = str(k.get("id_khoan", "")).strip(); ndk = (k.get("noi_dung_khoan") or "").strip()
        lines = []
        head = (f"{kid}. {ndk}" if kid else ndk).strip()
        if head: lines.append(head)
        for d in k.get("diem", []):
            did = str(d.get("id_diem", "")).strip(); ndd = (d.get("noi_dung") or "").strip()
            dl = (f"{did}) {ndd}" if did else ndd).strip()
            if dl: lines.append(dl)
        block = "\n".join(lines).strip()
        if block:
            blocks.append((int(kid) if kid.isdigit() else None, block))
    return intro, blocks

def _emit(chunks, doc_number, law_name, aid, chapter_ctx, text, clause_num, suffix):
    uid = f"{doc_number}_{aid}{suffix}"
    chunks.append({
        "id": uid,
        "text": text,
        "metadata": {
            "unique_article_id": uid,
            "article_id": aid,
            "doc_number": doc_number,
            "title": law_name,
            "context": {"part": None, "chapter": chapter_ctx, "section": None},
            "clause_num": clause_num,
        },
    })

def chunk_one(law_json: dict, doc_number: str, law_name: str, max_chars: int = None):
    if max_chars is None:
        max_chars = MAX_ARTICLE_CHARS
    chunks, seen = [], set()
    for ch in law_json.get("danh_sach_chuong", []):
        ch_id = (ch.get("id_chuong") or "").strip()
        ch_title = (ch.get("tieu_de_chuong") or "").strip()
        chapter_ctx = (f"{ch_id} {ch_title}".strip()) or None
        if ch_id.lower().startswith("chương mở đầu"):
            chapter_ctx = None
        for dieu in ch.get("danh_sach_dieu", []):
            aid = re.sub(r"\s+", " ", (dieu.get("id_dieu") or "").strip())
            if not aid or aid in seen:
                continue
            seen.add(aid)
            tieu_de = re.sub(r"\s+", " ", (dieu.get("tieu_de") or "").strip())
            intro, blocks = article_blocks(dieu)
            body_full = "\n".join(([intro] if intro else []) + [b for _, b in blocks]).strip()
            text_full = (f"{tieu_de}\n{body_full}".strip() if tieu_de else body_full).strip()
            if not text_full:
                continue

            # Đủ ngắn (hoặc không có Khoản để tách) -> 1 chunk cấp Điều, clause_num=None
            if len(text_full) <= max_chars or not blocks:
                if len(text_full) > max_chars:
                    print(f"  ⚠️ {doc_number} {aid}: {len(text_full)} ký tự, không tách được (thiếu Khoản).")
                _emit(chunks, doc_number, law_name, aid, chapter_ctx, text_full, None, "")
                continue

            # Dài -> gom các Khoản nguyên vẹn thành nhiều mảnh < max_chars
            budget = max(800, max_chars - len(tieu_de) - 2)
            pieces, cur, cur_len, cur_start = [], [], 0, None
            for num, b in blocks:
                if cur and cur_len + len(b) + 1 > budget:
                    pieces.append((cur_start, cur)); cur, cur_len, cur_start = [], 0, None
                if cur_start is None:
                    cur_start = num
                cur.append(b); cur_len += len(b) + 1
            if cur:
                pieces.append((cur_start, cur))

            for idx, (start, blks) in enumerate(pieces):
                body = "\n".join(blks)
                if idx == 0 and intro:
                    body = intro + "\n" + body
                text = (f"{tieu_de}\n{body}".strip() if tieu_de else body).strip()
                suffix = f"_K{start}" if start is not None else f"_p{idx + 1}"
                _emit(chunks, doc_number, law_name, aid, chapter_ctx, text, start, suffix)
                if len(text) > max_chars:
                    print(f"  ⚠️ {doc_number} {aid}{suffix}: 1 Khoản dài {len(text)} ký tự.")
    return chunks

print("Đã định nghĩa hàm chunking (tự tách Điều dài theo Khoản).")

## Cell 5 — Chạy chunking → corpus + manifest + bảng rà số hiệu

In [ ]:
import csv

all_chunks, manifest, review = [], {}, []
id_seen = set()

for fp in files:
    with open(fp, encoding="utf-8") as f:
        law = json.load(f)
    fname = os.path.basename(fp)
    dn, conf = extract_doc_number(law, fname)
    name = clean_law_name(law.get("ten_luat", "")) or dn
    dtype = infer_document_type(dn, name)
    cks = chunk_one(law, dn, name)
    kept = 0
    for c in cks:
        if c["id"] in id_seen:
            continue
        id_seen.add(c["id"]); all_chunks.append(c); kept += 1
    manifest[dn] = {"doc_id": dn, "document_type": dtype, "law_name": name,
                    "btc_standard_string": build_btc_string(dn, dtype, name)}
    review.append((fname, dn, conf, dtype, kept, name))

with open(REVIEW_OUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f); w.writerow(["file","doc_number","confidence","document_type","num_chunks","law_name"])
    for r in sorted(review, key=lambda r: (r[2] != "low", r[1])):
        w.writerow(r)

low  = [r for r in review if r[2] == "low"]
zero = [r for r in review if r[4] == 0]
print(f"✅ {len(all_chunks)} chunk từ {len(review)} văn bản | {len(manifest)} số hiệu trong manifest")
print(f"   Review CSV: {REVIEW_OUT}")
if zero: print(f"   ⚠️ {len(zero)} văn bản ra 0 chunk:", [r[0] for r in zero][:10])
if low:
    print(f"   ⚠️ {len(low)} văn bản số hiệu confidence=LOW (rà tay):")
    for r in low[:20]:
        print(f"      {r[1]:<22} <- {r[0]}")

## Cell 6 — Kiểm tra mắt thường vài chunk (đối chiếu format corpus gốc)

In [ ]:
print("Tổng chunk:", len(all_chunks))
print(json.dumps(all_chunks[0], ensure_ascii=False, indent=2)[:900])
print("\n--- một chunk có chương + khoản ---")
for c in all_chunks:
    if c["metadata"]["context"]["chapter"] and "\n1." in c["text"]:
        print(json.dumps(c, ensure_ascii=False, indent=2)[:1100]); break

## Cell 7 — Kiểm tra GPU + nạp embedding model

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBEDDING_MODEL, device=device)
model.max_seq_length = MAX_SEQ_LEN          # <-- chốt seq len, tránh nổ VRAM ở 8192
print("Loaded", EMBEDDING_MODEL, "| dim =", model.get_sentence_embedding_dimension(),
      "| max_seq_length =", model.max_seq_length)
assert model.get_sentence_embedding_dimension() == EMBEDDING_DIM, "Dim không khớp 1024!"

## Cell 8 — Embed toàn bộ chunk (normalize=True)

In [ ]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()

texts = [c["text"] for c in all_chunks]
# encode tự sắp xếp theo độ dài để giảm padding; batch nhỏ + max_seq_length đã chốt -> không OOM.
vectors = model.encode(
    texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
    show_progress_bar=True, convert_to_numpy=True,
)
print("Embedded:", vectors.shape)
# Nếu VẪN OOM: hạ BATCH_SIZE xuống 4 ở Cell 2, Restart kernel, chạy lại từ đầu.

## Cell 9 — Tạo collection `law_2026` + upsert

- Nếu `law_2026` chưa tồn tại → tạo mới (`size=1024`, `distance=DOT`).
- Nếu đã tồn tại và `RECREATE=False` → chỉ upsert (idempotent).
- **Chặn cứng**: không bao giờ thao tác lên `vietnamese_laws`.


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

assert COLLECTION_NAME != "vietnamese_laws", "Không được ghi vào collection cũ!"
client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"), timeout=120)

existing = [c.name for c in client.get_collections().collections]
dist = Distance.DOT if QDRANT_DISTANCE.lower() == "dot" else Distance.COSINE

if COLLECTION_NAME in existing and RECREATE:
    client.delete_collection(COLLECTION_NAME)
    print("♻️ Đã xoá collection cũ để tạo lại.")
if COLLECTION_NAME not in [c.name for c in client.get_collections().collections]:
    client.create_collection(COLLECTION_NAME,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=dist))
    print(f"🆕 Tạo collection '{COLLECTION_NAME}' (dim={EMBEDDING_DIM}, {QDRANT_DISTANCE}).")
else:
    print(f"ℹ️ '{COLLECTION_NAME}' đã tồn tại -> chỉ UPSERT.")

points = []
for c, vec in zip(all_chunks, vectors):
    m = c["metadata"]
    payload = {
        "chunk_id": c["id"],
        "unique_article_id": m["unique_article_id"],
        "article_id": m["article_id"],
        "doc_id": m["doc_number"],
        "title": m["title"],
        "text": c["text"],
        "context": m["context"],
        "clause_num": m["clause_num"],
        "original_content": c["text"],
    }
    pid = str(uuid.uuid5(UUID_NAMESPACE, m["unique_article_id"]))
    points.append(PointStruct(id=pid, vector=vec.tolist(), payload=payload))

for i in range(0, len(points), 256):
    client.upsert(collection_name=COLLECTION_NAME, points=points[i:i+256])
    print(f"  upsert {min(i+256, len(points))}/{len(points)}")
print("✅ Upsert xong.")

## Cell 10 — Kiểm chứng: số điểm + thử 1 truy vấn

In [ ]:
info = client.get_collection(COLLECTION_NAME)
print(f"'{COLLECTION_NAME}' points_count = {info.points_count}")

qv = model.encode(["Điều kiện hỗ trợ doanh nghiệp nhỏ và vừa"],
                  normalize_embeddings=True)[0].tolist()
hits = client.query_points(collection_name=COLLECTION_NAME, query=qv, limit=3, with_payload=True).points
for h in hits:
    p = h.payload
    print(f"  {h.score:.3f} | {p['doc_id']} {p['article_id']} | {p['text'][:70]}...")

## Cell 11 — Lưu corpus_clean.json + law_manifest.json (tải về từ Output)

In [ ]:
with open(CORPUS_OUT, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
with open(MANIFEST_OUT, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("💾 Đã ghi:")
for p in (CORPUS_OUT, MANIFEST_OUT, REVIEW_OUT):
    print("  ", p, f"({os.path.getsize(p)/1e6:.2f} MB)")
print("\nTrên Kaggle pipeline: đặt COLLECTION_NAME='law_2026', dùng 2 file trên cho BM25 + manifest.")